# NeMo guardrails with Langchain

Install dependencies and import libraries


In [ ]:
%pip install -U langchain-nvidia-ai-endpoints

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from google.colab import userdata
NVIDIA_API_KEY = userdata.get('apikey')

Instantiate the guard model

In [ ]:
guard_model = ChatNVIDIA(model="nvidia/llama-3.1-nemoguard-8b-content-safety")

Instantiate the main LLM

In [ ]:
llm = ChatNVIDIA(
    model="meta/llama3-8b-instruct",
    api_key = NVIDIA_API_KEY,
    temperature = 0.5,
    )

Define the prompt and chain for the main LLM

In [ ]:
prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("user", "{input}")
    ])
chain = prompt | main_llm | StrOutputParser()    # convert the output to a string so that we can do a string search for "unsafe"

In [ ]:
user_input = "How can I kill a Linux process"
#user_input = "How can I kill a horse"

Pass the user input through the guard model

In [ ]:
safety_check = guard_model.invoke(user_input)

Invoke the LLM chain only if it passes the safety filter

In [ ]:
if "unsafe" in safety_check.choices[0].message:
  print("I can't help you with that")
else:
  response = chain.invoke({"input": user_input})
  print(response.content)